# **LangChain Assignment**

**1. Install**


In [8]:
!pip install -U langchain langchain-core langchain-community langchain-groq faiss-cpu langchain-huggingface

**2. API Key**

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "Your Api Key"

**3. LLM Call**

In [12]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

response = llm.invoke("Explain AI in one line")
print(response.content)

Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making.


**4. Prompt Template**

In [13]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("Explain {topic} in simple terms")

print(llm.invoke(prompt.format(topic="Machine Learning")).content)

**What is Machine Learning.** 
Machine learning is a type of artificial intelligence (AI) that allows computers to learn and improve their performance on a task without being explicitly programmed. It's like teaching a child to recognize objects - you show them examples, and they learn to identify them on their own.

**How Does it Work.** 
Here's a simplified overview:

1. **Data Collection**: Gather lots of data related to the task you want the computer to learn (e.g., images, text, or sounds).
2. **Pattern Recognition**: The computer looks for patterns in the data, like a child finding similarities between objects.
3. **Model Training**: The computer uses the patterns to create a model that can make predictions or decisions.
4. **Testing and Refining**: The model is tested with new, unseen data to see how well it performs. If it makes mistakes, the model is refined and improved.

**Types of Machine Learning:**

1. **Supervised Learning**: The computer is shown labeled examples (e.g.,

**5. Chain**

In [14]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

print(chain.invoke({"topic": "Chatbots"}))

**What is a Chatbot.** 
A chatbot is a computer program that can have a conversation with you, either by text or voice. It's like a robot that can talk to you and answer your questions.

**How does it work.** 
Imagine you're chatting with a friend, but instead of a person, it's a computer program. You type or say something, and the chatbot responds. It uses special algorithms to understand what you're saying and give you a helpful answer.

**What can chatbots do.** 
Chatbots can:

1. **Answer questions**: They can provide information on a wide range of topics, like customer support, news, or entertainment.
2. **Help with tasks**: They can assist with things like booking flights, making reservations, or even helping you with your shopping.
3. **Entertain**: They can play games, tell jokes, or even create stories.
4. **Learn and improve**: Some chatbots can learn from their conversations and get better over time.

**Where can you find chatbots.** 
You can find chatbots in many places, su

**6. Memory **

In [15]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

prompt_mem = ChatPromptTemplate.from_messages([
    ("system", "You are helpful"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain_mem = prompt_mem | llm | StrOutputParser()

store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chat = RunnableWithMessageHistory(
    chain_mem,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "1"}}

print(chat.invoke({"input": "Hi my name is Disha"}, config=config))
print(chat.invoke({"input": "What is my name?"}, config=config))

Hello Disha, it's nice to meet you. Is there something I can help you with or would you like to chat?
Your name is Disha. You told me that when we started chatting.


**7. Agent**

In [17]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# Define tool (IMPORTANT: docstring required)
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers and return the result."""
    return a * b

# Create agent
agent = create_react_agent(
    model=llm,
    tools=[multiply]
)

# Run agent
result = agent.invoke({
    "messages": [
        ("human", "Multiply 5 and 6")
    ]
})

# Print final answer
print(result["messages"][-1].content)

/tmp/ipykernel_5896/3147066525.py:11: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


The result of multiplying 5 and 6 is 30.


**8. Vector Store**

In [18]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

docs = [
    Document(page_content="LangChain helps build LLM apps"),
    Document(page_content="FAISS is used for similarity search")
]

emb = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

db = FAISS.from_documents(docs, emb)

res = db.similarity_search("What is FAISS?")
print(res[0].page_content)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS is used for similarity search
